In [48]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import LabelEncoder
import sys
import os
sys.path.append(os.path.abspath('..'))

Загрузим данные и сделаем один большой датасет. Почему то у меня не подгрузилось с других ноутбуков и файлов, поэтому загрузим все данные заново.

In [49]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
category_name_t = pd.read_csv('../data/raw/product_category_name_translation.csv')
review = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

In [50]:
#Преобразуем строки в даты 
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

#Создаем признаки, которые были в EDA
orders['is_delivered'] = orders['order_delivered_customer_date'].notna().astype(int)
orders['is_approved'] = orders['order_approved_at'].notna().astype(int)

#Время доставки в днях
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['delivery_days'] = orders['delivery_days'].fillna(-1)

#Задержка доставки
orders['is_delayed'] = (
    (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']) & 
    (orders['is_delivered'] == 1)
).astype(int)

In [51]:
payments_agg = order_payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_installments': 'max',
    'payment_sequential': 'max'
}).reset_index()

# Собираем главный датасет
df_model = (orders
            .merge(order_items, on='order_id', how='inner')
            .merge(products, on='product_id', how='left')
            .merge(payments_agg, on='order_id', how='left')
            .merge(customers, on='customer_id', how='left')
            .merge(sellers, on='seller_id', how='left')
)

Сделаем подготовку для расчета прибыльности

In [52]:
#Параметры прибыльности
COST_PER_GRAM = 0.02  #Себестоимость за грамм (в бразильских реалах)
MIN_MARGIN = 0.20     #Минимальная маржа 20%

#Расчет
df_model['estimated_cost'] = df_model['product_weight_g'].fillna(0) * COST_PER_GRAM
df_model['total_cost'] = df_model['estimated_cost'] + df_model['freight_value']
df_model['profit'] = df_model['price'] - df_model['total_cost']
df_model['profit_margin'] = df_model['profit'] / (df_model['total_cost'] + 1)

# 1 - прибыльно, 0 - убыточно
df_model['target'] = (df_model['profit_margin'] >= MIN_MARGIN).astype(int)

print(f" Доля прибыльных заказов: {df_model['target'].mean():.1%}")

 Доля прибыльных заказов: 71.5%


In [53]:
#Числовые признаки
numeric_features = [
    'price', 'freight_value', 'payment_value', 'payment_installments',
    'product_weight_g', 'product_photos_qty',
    'is_delivered', 'is_approved', 'delivery_days', 'is_delayed'
]

#Кодирование категорий (для штатов и категорий товаров)
#Используем простые метки
le_state = LabelEncoder()
df_model['customer_state_enc'] = le_state.fit_transform(df_model['customer_state'].fillna('unknown'))

le_cat = LabelEncoder()
df_model['product_category_enc'] = le_cat.fit_transform(df_model['product_category_name'].fillna('unknown'))

#Итоговый список признаков
feature_cols = numeric_features + ['customer_state_enc', 'product_category_enc']

#Заполняем пропуски нулями (на всякий случай)
X = df_model[feature_cols].fillna(0)
y = df_model['target']

print(f"Количество признаков: {len(feature_cols)}")


Количество признаков: 12


In [54]:
#разделим по дате
df_model['order_purchase_timestamp'] = pd.to_datetime(df_model['order_purchase_timestamp'])
df_sorted = df_model.sort_values('order_purchase_timestamp')

X_sorted = df_sorted[feature_cols].fillna(0)
y_sorted = df_sorted['target']

#80% данных на обучение, 20% на тест
split_idx = int(len(X_sorted) * 0.8)

X_train, X_test = X_sorted.iloc[:split_idx], X_sorted.iloc[split_idx:]
y_train, y_test = y_sorted.iloc[:split_idx], y_sorted.iloc[split_idx:]

print(f"Train period: {df_sorted.iloc[0]['order_purchase_timestamp'].date()} — {df_sorted.iloc[split_idx-1]['order_purchase_timestamp'].date()}")
print(f"Test period: {df_sorted.iloc[split_idx]['order_purchase_timestamp'].date()} — {df_sorted.iloc[-1]['order_purchase_timestamp'].date()}")

Train period: 2016-09-04 — 2018-05-24
Test period: 2018-05-24 — 2018-09-03


Инициализируем и обучим нашу первую модель >:)

In [55]:
#Инициализация и обучение простой модели
baseline_model = RandomForestClassifier(n_estimators=50, random_state=2804, n_jobs=-1)

baseline_model.fit(X_train, y_train)

#Предсказание вероятностей
y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]

#Оценка
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Результат baseline-модели")
print(f"ROC-AUC Score: {roc_auc:.4f}")

importances = pd.Series(baseline_model.feature_importances_, index=feature_cols)
print("\nTop 5 важных признаков:")

print(importances.sort_values(ascending=False).head())

Результат baseline-модели
ROC-AUC Score: 0.9986

Top 5 важных признаков:
product_weight_g        0.355314
price                   0.315206
payment_value           0.119111
freight_value           0.112898
product_category_enc    0.037755
dtype: float64


Судя по ошеломительному результату, модель поняла, по какой формуле мы считаем прибыльность. Надо что-то менять. 

А так же судя по выводу важных признаков, 4 из 5 топ-признаков - это прямые компоненты формулы прибыли. Модель не нашла скрытые паттерны, она просто запомнила формулу(

Поэтому лишим ее такой возможности и дадим другие признаки (уберем вес, цену и доставку)

In [56]:
df_model['order_purchase_timestamp'] = pd.to_datetime(df_model['order_purchase_timestamp'], errors='coerce')
df_model['order_delivered_customer_date'] = pd.to_datetime(df_model['order_delivered_customer_date'], errors='coerce')
df_model['order_estimated_delivery_date'] = pd.to_datetime(df_model['order_estimated_delivery_date'], errors='coerce')

#Создаем логистические признаки
df_model['is_delivered'] = df_model['order_delivered_customer_date'].notna().astype(int)
df_model['is_approved'] = df_model['order_approved_at'].notna().astype(int)
df_model['delivery_days'] = (df_model['order_delivered_customer_date'] - df_model['order_purchase_timestamp']).dt.days
df_model['delivery_days'] = df_model['delivery_days'].fillna(-1)
df_model['is_delayed'] = ((df_model['order_delivered_customer_date'] > df_model['order_estimated_delivery_date']) & 
    (df_model['is_delivered'] == 1)).astype(int)

#Временные признаки
df_model['order_hour'] = df_model['order_purchase_timestamp'].dt.hour
df_model['order_dayofweek'] = df_model['order_purchase_timestamp'].dt.dayofweek

#Кодирование категорий
le_cat = LabelEncoder()
df_model['product_category_enc'] = le_cat.fit_transform(df_model['product_category_name'].fillna('unknown'))

le_cust = LabelEncoder()
df_model['customer_state_enc'] = le_cust.fit_transform(df_model['customer_state'].fillna('unknown'))

le_sell = LabelEncoder()
df_model['seller_state_enc'] = le_sell.fit_transform(df_model['seller_state'].fillna('unknown'))

#Убираем price, freight_value и weight
honest_features = [
    'payment_value', 'payment_installments', #Финансы (кроме цены товара)
    'product_photos_qty', 'product_category_enc', #О товаре
    'customer_state_enc', 'seller_state_enc', #География
    'is_delivered', 'is_approved', 'delivery_days', 'is_delayed', #Логистика
    'order_hour', 'order_dayofweek' #Время
]

print(f"Используем {len(honest_features)} новых признаков")

X_honest = df_model[honest_features].fillna(0)
y = df_model['target']

#Временной сплит
df_sorted = df_model.sort_values('order_purchase_timestamp')
X_sorted = X_honest.iloc[df_sorted.index]
y_sorted = y.iloc[df_sorted.index]

split_idx = int(len(X_sorted) * 0.8)
X_train, X_test = X_sorted.iloc[:split_idx], X_sorted.iloc[split_idx:]
y_train, y_test = y_sorted.iloc[:split_idx], y_sorted.iloc[split_idx:]

#Обучение
model_honest = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
model_honest.fit(X_train, y_train)

#Предсказание
y_pred_proba = model_honest.predict_proba(X_test)[:, 1]
roc_auc_honest = roc_auc_score(y_test, y_pred_proba)

#Результат
print("Результат нового теста")
print(f"ROC-AUC Score: {roc_auc_honest:.4f}")

importances = pd.Series(model_honest.feature_importances_, index=honest_features)
print("\n Top 5 важных признаков:")
print(importances.sort_values(ascending=False).head())

Используем 12 новых признаков
Результат нового теста
ROC-AUC Score: 0.7851

 Top 5 важных признаков:
payment_value           0.281286
product_category_enc    0.195534
order_hour              0.107824
delivery_days           0.106878
customer_state_enc      0.073057
dtype: float64


Вот теперь модель уже начинает искать какие-то паттерны, что очень хорошо

Важность новых признаков:




| Признак                  | Важность | Бизнес-инсайт                                                                    |
|--------------------------|----------|----------------------------------------------------------------------------------|
| payment_value            | 28.1%    | Заказы с большей суммой чаще прибыльны  |
| product_category_enc     | 19.5%    | Некоторые категории товаров маржинальнее других          |
| order_hour               | 10.8%    | Время заказа влияет     |
| delivery_days            | 10.7%    | Чем дольше доставка, тем ниже прибыль (логистические издержки, риск отмен)       |
| customer_state_enc       | 7.3%     | География важна: в одних штатах логистика дешевле, в других - выше конкуренция   |

Дальше буду думать как еще можно поиграть с признаками